# Set Permutation Likelihood

Experimental likelihood for unordered recall multisets.

The event must be specified: a **complete sequence** contains the observed items
in any order followed by STOP; a **prefix** contains those items first, leaving
subsequent recall unspecified. Use `RecallSetLikelihoodLoss` for the former and
`RecallPrefixLikelihoodLoss` for the latter.

For k observed items with multiplicities m_i, there are
$D = k! / \prod_i m_i!$ distinct orderings. The estimate is
$\widehat P = D\,S^{-1}\sum_s P(\pi_s)$, with uniform sampled permutations.
Complete-sequence probabilities include one STOP, even if the stored row is full.
An empty complete sequence scores immediate STOP; an empty prefix has probability one.

The implementation computes the estimate in log space. The probability estimate
is unbiased, but its logarithm generally is not. Sampling error can produce
estimates above one; the implementation does not clip them. Arbitrary event masks
are not part of this interface.

```python
from functools import partial
from jaxcmr.experimental.set_permutation_likelihood import (
    RecallSetLikelihoodLoss,
    RecallPrefixLikelihoodLoss,
)

# Supply this constructor as loss_fn_cls when creating an ordinary fitter.
loss_factory = partial(RecallSetLikelihoodLoss, permutation_count=100, seed=0)
loss = loss_factory(model_factory_cls, data, features)
```

The default is 50 permutations and seed zero. Sampled orders stay fixed across
parameter candidates. The model still conditions on each preceding response;
only the recorded ordering is marginalized. Repeated items retain multiplicity.
The model must permit any repeated responses in the observations.

Increasing the permutation count reduces Monte Carlo variability at additional
cost. Compare estimates across seeds when assessing that uncertainty. A sequence
likelihood remains preferable when the recorded order is part of the observation
being modeled. Marginalizing order does not make an order-sensitive model exchangeable.

Historical scores from the former average-only permutation loss omit the
number-of-orderings factor and do not represent this definition. Recalculate them
before comparing with new scores. Saved research results are not updated automatically.

See the [API reference](../../../guide/reference/experimental-permutation.qmd).
